# Incremental gait information

**Purpose:** Compare prescribed-condition prediction (C) with combined condition-and-gait prediction (C+G).

**Required inputs:** Prepared Baček HDF5; 04 foundational results ZIP.

**Outputs:** Matched held-out predictions and model-comparison records.

**Usage:** Run after 04. Files are read from the notebook’s current working directory unless specified in its configuration.


In [ ]:
# 1 — Imports, fixed configuration and basic helpers
from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import io
import json
import os
import platform
import sys
import time
import zipfile

import h5py
import numpy as np
import scipy
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

H5 = Path('bacek_clean_common_angles_metabolic_dataset.h5')
REFERENCE_ZIP = Path('04_Bacek_Foundational_Predictive_Models_results.zip')
OUT = Path('05_Bacek_Incremental_Gait_Information_results')
ARCHIVE = Path('05_Bacek_Incremental_Gait_Information_results.zip')
EXPECTED_SHA = 'c73ce561e94de52f1dc86128829c43ef536c4cf6b5f4aa9af1ea060409221fa3'
CHANNELS = ('hip_left', 'hip_right', 'knee_left', 'knee_right',
            'ankle_left', 'ankle_right', 'pelvis_sagittal',
            'pelvis_frontal', 'pelvis_transverse')
ALPHAS = (0.01, 0.1, 1.0, 10.0, 100.0, 1000.0)
PCA_COUNTS = (5, 10, 20, 40)
SPEEDS = (1.1, 0.8, 0.4)
CADENCES = (0.90, 0.95, 1.00, 1.10, 1.20)
OUTER_FOLDS = {
    1: ('S08','S10','S16','S21'),
    2: ('S11','S13','S17','S20'),
    3: ('S01','S02','S05','S18'),
    4: ('S03','S07','S12','S14','S19'),
    5: ('S04','S06','S09','S15'),
}
EXPECTED_COUNTS = (55, 53, 56, 65, 55)
FOLD_OF = {s: k for k, people in OUTER_FOLDS.items() for s in people}


def check(condition, message):
    if not bool(condition):
        raise AssertionError(message)


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def write_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)


def read_reference_csv(archive, name):
    with archive.open(name) as f:
        return list(csv.DictReader(io.TextIOWrapper(f, encoding='utf-8')))


def rmse(y_true, y_pred):
    error = np.asarray(y_pred, dtype=np.float64) - np.asarray(y_true, dtype=np.float64)
    return float(np.sqrt(np.mean(error**2, dtype=np.float64)))


def performance(y_true, y_pred):
    actual = np.asarray(y_true, dtype=np.float64)
    predicted = np.asarray(y_pred, dtype=np.float64)
    error = predicted - actual
    denom = float(np.sum((actual - actual.mean())**2, dtype=np.float64))
    return {
        'n_trials': int(len(actual)),
        'RMSE_W_per_kg': rmse(actual, predicted),
        'MAE_W_per_kg': float(np.mean(np.abs(error), dtype=np.float64)),
        'bias_W_per_kg': float(np.mean(error, dtype=np.float64)),
        'R2': float(1 - np.sum(error**2, dtype=np.float64)/denom) if denom > 0 else None,
    }

print('Python:', platform.python_version(),
      '| NumPy:', np.__version__, '| scikit-learn:', sklearn.__version__)

In [ ]:
# 2 — Read the verified Notebook 04 evidence; preserve C and G predictions
check(REFERENCE_ZIP.is_file(),
      'Notebook 04 results ZIP must be placed beside this notebook')
reference_sha = sha256_file(REFERENCE_ZIP)
with zipfile.ZipFile(REFERENCE_ZIP) as z:
    reference_summary = json.loads(z.read('run_summary.json'))
    reference_ledger = read_reference_csv(z, 'eligible_trial_ledger.csv')
    reference_outer = read_reference_csv(z, 'outer_fold_manifest.csv')
    reference_inner = read_reference_csv(z, 'inner_fold_manifest.csv')
    reference_predictions = read_reference_csv(z, 'held_out_predictions.csv')
    reference_checks = read_reference_csv(z, 'verification_checks.csv')
    reference_metrics = read_reference_csv(z, 'pooled_metrics.csv')

check(reference_summary['status'] == 'PASS', 'Notebook 04 did not report PASS')
check(reference_summary['source_sha256'] == EXPECTED_SHA, 'Notebook 04 source mismatch')
check(reference_summary['total_primary_trials'] == 284, 'Notebook 04 population mismatch')
check(all(x['status'] == 'PASS' for x in reference_checks),
      'Notebook 04 reported a failed verification check')
check(len(reference_ledger) == 284 and len(reference_inner) == 20 and
      len(reference_outer) == 5, 'Notebook 04 manifest completeness mismatch')
check({r['trial_uid'] for r in reference_ledger} ==
      {r['trial_uid'] for r in reference_predictions if r['procedure'] == 'B2'},
      'Notebook 04 B2 predictions do not cover the eligible ledger')

reference_by_uid = {r['trial_uid']: r for r in reference_ledger}
check(len(reference_by_uid) == 284, 'Duplicate trial UID in Notebook 04')
ref_predictions = {}
for procedure in ('B2', 'G_SELECTED'):
    selected = {r['trial_uid']: r for r in reference_predictions
                if r['procedure'] == procedure}
    check(len(selected) == 284 and set(selected) == set(reference_by_uid),
          f'Notebook 04 {procedure} prediction coverage mismatch')
    ref_predictions[procedure] = selected

for procedure in ('B2', 'G_SELECTED'):
    for uid, pred in ref_predictions[procedure].items():
        ledger = reference_by_uid[uid]
        check(pred['subject_id'] == ledger['subject_id'] and
              pred['trial_id'] == ledger['trial_id'] and
              int(pred['outer_fold']) == int(ledger['outer_fold']) and
              float(pred['observed_W_per_kg']) == float(ledger['observed_W_per_kg']) and
              np.isfinite(float(pred['predicted_W_per_kg'])),
              f'Notebook 04 {procedure} identity/target/prediction discrepancy: {uid}')

for r in reference_outer:
    fold = int(r['outer_fold'])
    expected_test = set(OUTER_FOLDS[fold])
    expected_dev = set(FOLD_OF) - expected_test
    check(set(r['test_participants'].split(';')) == expected_test and
          set(r['development_participants'].split(';')) == expected_dev and
          int(r['n_test_trials']) == EXPECTED_COUNTS[fold-1],
          f'Notebook 04 outer manifest mismatch: {fold}')

print('Notebook 04 input verified: 284 matched C/B2 and G predictions; 20 inner folds')

In [ ]:
# 3 — Read corrected Baček trials and construct the exact C, W and D inputs

def features_from_trial(trial):
    waveforms = []
    n_cycles = None
    for channel in CHANNELS:
        check(channel in trial, f'Missing gait channel: {channel}')
        angle = np.asarray(trial[channel][()], dtype=np.float64)
        check(angle.ndim == 2 and angle.shape[1] == 101 and
              np.isfinite(angle).all(), f'Invalid angle data: {channel}')
        if n_cycles is None:
            n_cycles = angle.shape[0]
        check(angle.shape[0] == n_cycles, 'Inconsistent channel cycle counts')
        waveforms.append(angle.mean(axis=0, dtype=np.float64))
    mean_waveforms = np.stack(waveforms)
    descriptors = []
    for channel in mean_waveforms:
        for q in range(4):
            descriptors.append(float(np.trapezoid(
                channel[25*q:25*q+26], dx=1.0) / 25.0))
        descriptors.append(float(np.std(channel, ddof=0)))
        descriptors.append(float(np.max(channel) - np.min(channel)))
    return mean_waveforms.reshape(909), np.asarray(descriptors, dtype=np.float64), n_cycles

check(H5.is_file(), f'Corrected HDF5 not found: {H5}')
source_sha = sha256_file(H5)
check(source_sha == EXPECTED_SHA, 'Source checksum mismatch — do not continue')

rows, waveform_rows, descriptor_rows = [], [], []
with h5py.File(H5, 'r') as h5:
    check(set(h5['subjects']) == set(FOLD_OF), 'Unexpected source participants')
    for subject in sorted(h5['subjects']):
        for trial_id in sorted(h5['subjects'][subject]['trials']):
            if not (trial_id.startswith('T') and trial_id[1:].isdigit() and
                    1 <= int(trial_id[1:]) <= 15):
                continue
            trial = h5['subjects'][subject]['trials'][trial_id]
            uid = f'{subject}_{trial_id}'
            w, d, count = features_from_trial(trial)
            for attr in ('valid', 'comparison_eligible', 'unrestricted_comparison_eligible'):
                check(attr in trial.attrs and
                      np.asarray(trial.attrs[attr]).reshape(-1)[0] == 1,
                      f'Invalid source flag {uid}: {attr}')
            target = np.asarray(trial['metcost_mean'][()])
            check(target.size == 1 and np.issubdtype(target.dtype, np.number)
                  and np.isrealobj(target), f'Invalid scalar target: {uid}')
            value = float(target.reshape(-1)[0])
            check(np.isfinite(value) and value > 0, f'Invalid target value: {uid}')
            indices = np.asarray(trial['original_cycle_number'][()]).reshape(-1)
            check(len(indices) == count and np.isfinite(indices).all()
                  and np.all(indices == np.floor(indices))
                  and np.all(np.diff(indices) > 0), f'Invalid cycle provenance: {uid}')
            if uid == 'S06_T11':
                check(count == 85, 'Unexpected S06_T11 cycle count')
                continue
            check(count >= 100, f'Unexpected modelling eligibility: {uid}')
            block, cadence_index = divmod(int(trial_id[1:]) - 1, 5)
            rows.append({'trial_uid': uid, 'subject_id': subject,
                         'trial_id': trial_id, 'outer_fold': FOLD_OF[subject],
                         'retained_cycles': count, 'speed_m_s': SPEEDS[block],
                         'relative_cadence': CADENCES[cadence_index],
                         'condition_index': int(trial_id[1:]) - 1,
                         'observed_W_per_kg': value})
            waveform_rows.append(w)
            descriptor_rows.append(d)

check(len(rows) == 284 and len({r['trial_uid'] for r in rows}) == 284,
      'Expected 284 unique primary trial records')
check(sum(r['retained_cycles'] for r in rows) == 60757, 'Primary cycle count mismatch')
check([sum(r['outer_fold'] == f for r in rows) for f in OUTER_FOLDS]
      == list(EXPECTED_COUNTS), 'Outer trial counts mismatch')
for r in rows:
    old = reference_by_uid[r['trial_uid']]
    for field in ('subject_id', 'trial_id'):
        check(r[field] == old[field], f'Notebook 04 {field} discrepancy: {r["trial_uid"]}')
    for field in ('outer_fold', 'retained_cycles', 'condition_index'):
        check(r[field] == int(old[field]), f'Notebook 04 {field} discrepancy: {r["trial_uid"]}')
    for field in ('speed_m_s', 'relative_cadence', 'observed_W_per_kg'):
        check(r[field] == float(old[field]),
              f'Notebook 04 {field} discrepancy: {r["trial_uid"]}')

Xw = np.stack(waveform_rows)
Xd = np.stack(descriptor_rows)
Xc = np.eye(15, dtype=np.float64)[[r['condition_index'] for r in rows]]
Xcw = np.column_stack([Xc, Xw])
Xcd = np.column_stack([Xc, Xd])
y = np.array([r['observed_W_per_kg'] for r in rows], dtype=np.float64)
groups = np.array([r['subject_id'] for r in rows])
uids = np.array([r['trial_uid'] for r in rows])
check(Xcw.shape == (284, 924) and Xcd.shape == (284, 69), 'Combined feature shape mismatch')
check(np.isfinite(Xcw).all() and np.isfinite(Xcd).all(), 'Invalid combined features')
check(np.all(Xc.sum(axis=1) == 1), 'One-hot condition encoding incorrect')
print('Verified corrected source and matched Notebook 04 ledger: 284 trials')
print('Combined design matrices:', 'C+W', Xcw.shape, '| C+D', Xcd.shape)

## Combined-model selection


In [ ]:
# 4 — Define and verify all 36 complete C+G candidates

def candidate_grid():
    result = []
    for representation, size in (('W-ALL9', 909), ('D-ALL9', 54)):
        for alpha in ALPHAS:
            result.append({'representation': representation, 'alpha': alpha,
                           'n_components': None, 'n_features': 15+size})
    for count in PCA_COUNTS:
        for alpha in ALPHAS:
            result.append({'representation': 'P-ALL9', 'alpha': alpha,
                           'n_components': count, 'n_features': 15+count})
    return result


def candidate_key(candidate):
    return (candidate['representation'], candidate['n_components'], candidate['alpha'])


def combined_input(candidate):
    return Xcd if candidate['representation'] == 'D-ALL9' else Xcw


def combined_pipeline(candidate):
    # Condition columns are passed through unscaled. Only gait columns are learned/scaled.
    gait_steps = [('scale', StandardScaler())]
    if candidate['representation'] == 'P-ALL9':
        gait_steps.append(('pca', PCA(n_components=candidate['n_components'],
                                      whiten=False, random_state=2026)))
    gait_transform = Pipeline(gait_steps)
    combined_features = ColumnTransformer(
        [('conditions', 'passthrough', slice(0, 15)),
         ('gait', gait_transform, slice(15, None))], remainder='drop')
    return Pipeline([('features', combined_features),
                     ('ridge', Ridge(alpha=candidate['alpha'], fit_intercept=True))])


def choose_combined(scores):
    # Floating RMSE equality is exact; the specified tie order follows.
    return min(scores, key=lambda r:
               (r['inner_pooled_RMSE_W_per_kg'], r['n_features'],
                r['representation'], r['n_components'] or 0, r['alpha']))

CANDIDATES = candidate_grid()
check(len(CANDIDATES) == 36 and len({candidate_key(c) for c in CANDIDATES}) == 36,
      'C+G must contain 36 distinct candidates')
check({c['n_features'] for c in CANDIDATES} == {20, 25, 35, 55, 69, 924},
      'Unexpected combined predictor dimensions')
print('C+G search:', len(CANDIDATES), 'candidate configurations')

In [ ]:
# 5 — Execute C+G with the exact Notebook 04 participant partitions
started_at = datetime.now(timezone.utc).isoformat()
t0 = time.perf_counter()
inner_scores, pooled_scores, selections, new_predictions, model_timings = [], [], [], [], []
verified_folds = []

for fold, test_people in OUTER_FOLDS.items():
    test_idx = np.flatnonzero(np.isin(groups, test_people))
    dev_idx = np.flatnonzero(~np.isin(groups, test_people))
    check(len(test_idx) == EXPECTED_COUNTS[fold-1], 'Outer test count mismatch')
    check(set(groups[test_idx]).isdisjoint(groups[dev_idx]), 'Outer participant overlap')

    partition_rows = sorted((r for r in reference_inner if int(r['outer_fold']) == fold),
                            key=lambda r: int(r['inner_fold']))
    check(len(partition_rows) == 4, f'Expected four inner folds for outer {fold}')
    inner_indices = []
    validation_covered = []
    for inner_number, manifest in enumerate(partition_rows, 1):
        expected_train = set(manifest['train_participants'].split(';'))
        expected_valid = set(manifest['validation_participants'].split(';'))
        train_idx = np.flatnonzero(np.isin(groups, list(expected_train)))
        valid_idx = np.flatnonzero(np.isin(groups, list(expected_valid)))
        check(int(manifest['inner_fold']) == inner_number and
              set(groups[train_idx]) == expected_train and
              set(groups[valid_idx]) == expected_valid,
              f'Notebook 04 inner participant IDs mismatch: outer {fold}, inner {inner_number}')
        check(expected_train.isdisjoint(expected_valid) and
              expected_train | expected_valid == set(groups[dev_idx]) and
              expected_train.isdisjoint(test_people) and
              expected_valid.isdisjoint(test_people),
              'Training/validation/outer participant overlap or incomplete partition')
        check(int(manifest['n_train_trials']) == len(train_idx) and
              int(manifest['n_validation_trials']) == len(valid_idx),
              'Saved inner trial count mismatch')
        inner_indices.append((train_idx, valid_idx))
        validation_covered.extend(valid_idx.tolist())
    check(sorted(validation_covered) == sorted(dev_idx.tolist()),
          'Each development trial must be validated exactly once')
    verified_folds.extend([{'outer_fold': fold, **r} for r in partition_rows])

    fold_candidates = []
    for candidate in CANDIDATES:
        x = combined_input(candidate)
        validation_errors = []
        for inner_number, (train_idx, valid_idx) in enumerate(inner_indices, 1):
            estimator = combined_pipeline(candidate)
            tic = time.perf_counter()
            estimator.fit(x[train_idx], y[train_idx])
            fitting_time = time.perf_counter() - tic
            tic = time.perf_counter()
            prediction = estimator.predict(x[valid_idx])
            prediction_time = time.perf_counter() - tic
            check(len(prediction) == len(valid_idx) and np.isfinite(prediction).all(),
                  'Invalid inner-validation predictions')
            validation_errors.extend((prediction - y[valid_idx]).tolist())
            inner_scores.append({
                'outer_fold': fold, 'inner_fold': inner_number,
                **candidate,
                'n_train_trials': len(train_idx), 'n_validation_trials': len(valid_idx),
                'train_RMSE_W_per_kg': rmse(y[train_idx], estimator.predict(x[train_idx])),
                'validation_RMSE_W_per_kg': rmse(y[valid_idx], prediction),
                'fit_seconds': fitting_time, 'prediction_seconds': prediction_time,
            })
        check(len(validation_errors) == len(dev_idx), 'Incomplete pooled candidate coverage')
        score = float(np.sqrt(np.mean(np.square(validation_errors), dtype=np.float64)))
        fold_candidates.append({**candidate, 'outer_fold': fold,
                                'inner_pooled_RMSE_W_per_kg': score})
    pooled_scores.extend(fold_candidates)
    choice = choose_combined(fold_candidates)
    selections.append({'outer_fold': fold, 'procedure': 'C+G', **choice})

    selected_matrix = combined_input(choice)
    model = combined_pipeline(choice)
    tic = time.perf_counter()
    model.fit(selected_matrix[dev_idx], y[dev_idx])
    final_fit_seconds = time.perf_counter() - tic
    tic = time.perf_counter()
    prediction = model.predict(selected_matrix[test_idx])
    final_prediction_seconds = time.perf_counter() - tic
    check(len(prediction) == len(test_idx) and np.isfinite(prediction).all(),
          'Invalid C+G outer-test predictions')
    model_timings.append({'outer_fold': fold, 'representation': choice['representation'],
                          'alpha': choice['alpha'], 'n_components': choice['n_components'],
                          'n_features': choice['n_features'],
                          'fit_seconds': final_fit_seconds,
                          'prediction_seconds': final_prediction_seconds,
                          'prediction_seconds_per_trial': final_prediction_seconds/len(test_idx),
                          'n_train_trials': len(dev_idx), 'n_predicted_trials': len(test_idx)})
    for i, idx in enumerate(test_idx):
        new_predictions.append({'procedure': 'C+G', 'outer_fold': fold,
                                'subject_id': rows[idx]['subject_id'],
                                'trial_id': rows[idx]['trial_id'],
                                'trial_uid': rows[idx]['trial_uid'],
                                'observed_W_per_kg': float(y[idx]),
                                'predicted_W_per_kg': float(prediction[i])})
    print(f'Outer fold {fold}: {len(test_idx)} held-out trials; '
          f'C+G selected {choice["representation"]}, alpha={choice["alpha"]}'
          + (f', PCA={choice["n_components"]}' if choice['n_components'] else ''))

elapsed_seconds = time.perf_counter() - t0
print(f'C+G completed: {len(new_predictions)} held-out predictions in {elapsed_seconds:.1f} s')

In [ ]:
# 6 — Pair C and C+G, preserve G for context, and verify all evidence
old_c = ref_predictions['B2']
old_g = ref_predictions['G_SELECTED']
new_cg = {r['trial_uid']: r for r in new_predictions}
check(len(new_cg) == 284, 'C+G trial keys are not unique')
check(set(new_cg) == set(old_c) == set(old_g) == set(uids),
      'Reference and C+G must use exactly the same 284 trials')

prediction_ledger = []
for procedure, lookup in (('C', old_c), ('G', old_g), ('C+G', new_cg)):
    for row in rows:
        uid = row['trial_uid']
        ref = lookup[uid]
        check(int(ref['outer_fold']) == row['outer_fold'] and
              ref['subject_id'] == row['subject_id'] and
              ref['trial_id'] == row['trial_id'] and
              float(ref['observed_W_per_kg']) == row['observed_W_per_kg'],
              f'Unmatched trial ID/target in {procedure}: {uid}')
        prediction_ledger.append({
            'procedure': procedure, 'outer_fold': row['outer_fold'],
            'subject_id': row['subject_id'], 'trial_id': row['trial_id'],
            'trial_uid': uid, 'observed_W_per_kg': row['observed_W_per_kg'],
            'predicted_W_per_kg': float(ref['predicted_W_per_kg'])})

checks = []
def verify(name, outcome):
    checks.append({'check': name, 'status': 'PASS' if bool(outcome) else 'FAIL'})
    check(outcome, name)

verify('source_matches_notebook04', source_sha == reference_summary['source_sha256'])
verify('primary_284_trial_ledger_matches_notebook04', len(rows) == 284)
verify('all_20_saved_inner_partitions_verified_and_reused', len(verified_folds) == 20)
verify('exactly_36_candidates', len(CANDIDATES) == 36)
verify('exactly_720_inner_candidate_fits', len(inner_scores) == 5*4*36)
verify('exactly_180_pooled_candidate_scores', len(pooled_scores) == 5*36)
verify('exactly_five_CG_selections', len(selections) == 5)
verify('each_selected_configuration_minimizes_inner_score', all(
    candidate_key(selected) == candidate_key(choose_combined(
        [r for r in pooled_scores if r['outer_fold'] == selected['outer_fold']]))
    for selected in selections))
verify('all_284_CG_predictions_finite', len(new_predictions) == 284 and all(
    np.isfinite(r['predicted_W_per_kg']) for r in new_predictions))
verify('one_prediction_per_trial_per_procedure', len(prediction_ledger) == 3*284 and
       len({(r['procedure'], r['trial_uid']) for r in prediction_ledger}) == 3*284)
verify('matched_C_G_CG_trials_and_targets', all(
    ref_predictions['B2'][uid]['observed_W_per_kg'] ==
    ref_predictions['G_SELECTED'][uid]['observed_W_per_kg'] and
    float(ref_predictions['B2'][uid]['observed_W_per_kg']) ==
    float(new_cg[uid]['observed_W_per_kg']) for uid in uids))
verify('C_predictions_unchanged_from_notebook04', all(
    float(old_c[uid]['predicted_W_per_kg']) == next(
        r['predicted_W_per_kg'] for r in prediction_ledger
        if r['trial_uid'] == uid and r['procedure'] == 'C') for uid in uids))
verify('G_predictions_unchanged_from_notebook04', all(
    float(old_g[uid]['predicted_W_per_kg']) == next(
        r['predicted_W_per_kg'] for r in prediction_ledger
        if r['trial_uid'] == uid and r['procedure'] == 'G') for uid in uids))

metrics, fold_metrics = [], []
for procedure in ('C', 'G', 'C+G'):
    subset = [r for r in prediction_ledger if r['procedure'] == procedure]
    metrics.append({'procedure': procedure, **performance(
        [r['observed_W_per_kg'] for r in subset],
        [r['predicted_W_per_kg'] for r in subset])})
    for fold in OUTER_FOLDS:
        part = [r for r in subset if r['outer_fold'] == fold]
        fold_metrics.append({'outer_fold': fold, 'procedure': procedure, **performance(
            [r['observed_W_per_kg'] for r in part],
            [r['predicted_W_per_kg'] for r in part])})

verify('all_metrics_finite', all(
    np.isfinite(r['RMSE_W_per_kg']) and np.isfinite(r['MAE_W_per_kg'])
    and np.isfinite(r['bias_W_per_kg']) and np.isfinite(r['R2']) for r in metrics))

# Independently compare reused C and G pooled metrics to the prior archive.
for new_name, old_name in (('C', 'B2'), ('G', 'G_SELECTED')):
    old_row = next(r for r in reference_metrics if r['procedure'] == old_name)
    new_row = next(r for r in metrics if r['procedure'] == new_name)
    for field in ('RMSE_W_per_kg', 'MAE_W_per_kg', 'bias_W_per_kg', 'R2'):
        check(abs(float(old_row[field]) - new_row[field]) < 1e-12,
              f'Notebook 04 {new_name} metric changed: {field}')
verify('reused_reference_metrics_match_notebook04', True)

metric_by_name = {r['procedure']: r for r in metrics}
primary_delta = metric_by_name['C']['RMSE_W_per_kg'] - metric_by_name['C+G']['RMSE_W_per_kg']
print('\nPooled held-out RMSE (W/kg):')
for r in metrics:
    print(f'  {r["procedure"]:4s} {r["RMSE_W_per_kg"]:.6f}')
print(f'Primary paired C − C+G RMSE: {primary_delta:+.6f} W/kg')
print('Positive difference favours C+G; uncertainty interval deferred to E5.2.')
print('Verification:', len(checks), 'checks passed')

In [ ]:
# 7 — Save all reusable E2.1 evidence and create the results archive
OUT.mkdir(parents=True, exist_ok=True)

notebook_filename = '05_Bacek_Incremental_Gait_Information.ipynb'
notebook_code_sha = None
if Path(notebook_filename).is_file():
    document = json.loads(Path(notebook_filename).read_text(encoding='utf-8'))
    joined_code = ''.join(''.join(c['source']) for c in document['cells']
                          if c['cell_type'] == 'code')
    notebook_code_sha = hashlib.sha256(joined_code.encode('utf-8')).hexdigest()

write_csv(OUT/'eligible_trial_ledger.csv', rows,
          ['trial_uid','subject_id','trial_id','outer_fold','retained_cycles',
           'speed_m_s','relative_cadence','condition_index','observed_W_per_kg'])
write_csv(OUT/'inner_fold_manifest.csv', reference_inner,
          ['outer_fold','inner_fold','train_participants','validation_participants',
           'outer_test_participants','n_train_trials','n_validation_trials'])
write_csv(OUT/'inner_candidate_scores.csv', inner_scores,
          ['outer_fold','inner_fold','representation','alpha','n_components','n_features',
           'n_train_trials','n_validation_trials','train_RMSE_W_per_kg',
           'validation_RMSE_W_per_kg','fit_seconds','prediction_seconds'])
write_csv(OUT/'pooled_candidate_scores.csv', pooled_scores,
          ['outer_fold','representation','alpha','n_components','n_features',
           'inner_pooled_RMSE_W_per_kg'])
write_csv(OUT/'selected_models.csv', selections,
          ['outer_fold','procedure','representation','alpha','n_components',
           'n_features','inner_pooled_RMSE_W_per_kg'])
write_csv(OUT/'held_out_predictions.csv', prediction_ledger,
          ['procedure','outer_fold','subject_id','trial_id','trial_uid',
           'observed_W_per_kg','predicted_W_per_kg'])
write_csv(OUT/'pooled_metrics.csv', metrics,
          ['procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg','bias_W_per_kg','R2'])
write_csv(OUT/'outer_fold_metrics.csv', fold_metrics,
          ['outer_fold','procedure','n_trials','RMSE_W_per_kg','MAE_W_per_kg',
           'bias_W_per_kg','R2'])
write_csv(OUT/'outer_model_timings.csv', model_timings,
          ['outer_fold','representation','alpha','n_components','n_features',
           'fit_seconds','prediction_seconds','prediction_seconds_per_trial',
           'n_train_trials','n_predicted_trials'])
write_csv(OUT/'verification_checks.csv', checks, ['check','status'])

summary = {
    'status': 'PASS' if all(r['status']=='PASS' for r in checks) else 'REVIEW_REQUIRED',
    'experiment': 'E2.1',
    'prediction_setting': 'new-person, internally held-out and exploratory-informed',
    'source_hdf5': H5.name, 'source_sha256': source_sha,
    'reference_notebook04_results': REFERENCE_ZIP.name,
    'reference_notebook04_zip_sha256': reference_sha,
    'reference_notebook04_code_sha256': reference_summary.get('notebook_code_sha256'),
    'target': 'corrected stored trial-level metcost_mean, net W/kg',
    'n_primary_trials': 284, 'n_primary_participants': 21,
    'n_candidates_per_outer_fold': 36, 'n_inner_folds_per_outer_fold': 4,
    'inner_splitter': 'exact four participant-grouped inner partitions saved in Notebook 04',
    'C': 'Notebook 04 B2 — reused exactly, no refitting',
    'G': 'Notebook 04 G_SELECTED — reused exactly for context, no refitting',
    'C_plus_G': '15 unscaled condition indicators + training-scaled gait W/D/P; PCA on gait only; Ridge with intercept',
    'alphas': list(ALPHAS), 'PCA_counts': list(PCA_COUNTS),
    'PCA': {'whiten': False, 'random_state': 2026},
    'tie_rule': 'exact pooled float64 inner RMSE, fewer total features, representation ID, components, alpha',
    'primary_comparison': 'RMSE(C) - RMSE(C+G), 284 matched held-out trials',
    'primary_delta_RMSE_W_per_kg': primary_delta,
    'uncertainty_interval_included': False,
    'sensitivity_285_included': False,
    'started_at_utc': started_at,
    'finished_at_utc': datetime.now(timezone.utc).isoformat(),
    'elapsed_seconds': elapsed_seconds,
    'software': {'python': platform.python_version(), 'numpy': np.__version__,
                 'scipy': scipy.__version__, 'scikit_learn': sklearn.__version__,
                 'h5py': h5py.__version__, 'os': platform.platform(),
                 'cpu_count': os.cpu_count(), 'kernel_python_executable': sys.executable},
    'notebook_filename': notebook_filename,
    'notebook_code_sha256': notebook_code_sha,
    'code_fingerprint_basis': 'concatenated code-cell sources, UTF-8',
    'verification': {r['check']: r['status'] for r in checks},
}
(OUT/'run_summary.json').write_text(json.dumps(summary, indent=2, allow_nan=False),
                                    encoding='utf-8')
files = ('eligible_trial_ledger.csv', 'inner_fold_manifest.csv',
         'inner_candidate_scores.csv', 'pooled_candidate_scores.csv',
         'selected_models.csv', 'held_out_predictions.csv', 'pooled_metrics.csv',
         'outer_fold_metrics.csv', 'outer_model_timings.csv',
         'verification_checks.csv', 'run_summary.json')
with zipfile.ZipFile(ARCHIVE, 'w', zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for name in files:
        z.write(OUT/name, arcname=name)
print('\nArchive created:', ARCHIVE, '| files:', len(files))
print('Recorded verification status:', summary['status'])